In [1]:
%load_ext autoreload
%autoreload 2

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "3" 
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["VECLIB_MAXIMUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"
os.environ["POLARS_MAX_THREADS"] = "8" # Polars можно дать чуть больше

import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Current GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA available: True
Current GPU: NVIDIA RTX A4000


In [2]:
import gc
import time
import numpy as np
import polars as pl
import kagglehub
from kagglehub import KaggleDatasetAdapter
from catboost import CatBoostClassifier, CatBoostRanker, Pool

/home/gavrishokni/.conda/envs/for_rec_sys_shad_course/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset_path = "thekabeton/ysda-recsys-2026-yambda-dataset/versions/3"
data_orig = kagglehub.dataset_load(KaggleDatasetAdapter.POLARS, dataset_path, "likes.parquet").collect()
artists = kagglehub.dataset_load(KaggleDatasetAdapter.POLARS, dataset_path, "artist_item_mapping_small.parquet").collect()

In [4]:
ENABLE_LOG = 1
def log(args):
    if ENABLE_LOG:
        print(args)

def recall_at_k(y_true_df, y_pred_df, k=100):
    top_k_pred = (
        y_pred_df.sort(["uid", "score"], descending=[False, True])
        .group_by("uid")
        .head(k)
    )
    hits = top_k_pred.join(y_true_df, on=["uid", "item_id"], how="inner")
    return hits.height / y_true_df.height

def get_latest_events(df: pl.DataFrame, n: int = 5000) -> pl.DataFrame:
    """
    Возвращает N самых последних событий на основе колонки timestamp.
    """
    return (
        df
        .sort("timestamp", descending=False) # Сортируем от старых к новым
        .tail(n)                             # Берем N последних строк (самые новые)
    )

def smart_sampling(df, max_events_per_user=3, target_rows=None):
    log(f"Исходный размер: {df.height} строк")

    # Сортируем по времени (новые сверху), чтобы head(N) взял последние события
    df_sorted = df.sort(["uid", "timestamp"], descending=[False, True])

    # Группируем по юзеру и берем топ N записей
    df_sampled = df_sorted.group_by("uid").head(max_events_per_user)
    
    log(f"Размер после ограничения истории ({max_events_per_user} на юзера): {df_sampled.height} строк")

    if target_rows and df_sampled.height > target_rows:
        fraction = target_rows / df_sampled.height
        unique_users = df_sampled.select("uid").unique()
        sampled_users = unique_users.sample(fraction=fraction, seed=42)
        df_sampled = df_sampled.join(sampled_users, on="uid", how="inner")
        log(f"Размер после жесткого ограничения: {df_sampled.height} строк")

    return df_sampled.sort("timestamp")

# data_orig_last5k = get_latest_events(data_orig, 5000)

data = smart_sampling(data_orig, max_events_per_user=5)
gc.collect()


data = data.sort('timestamp')
n = len(data)
split_1 = int(n * 0.60)
split_2 = int(n * 0.80)

print(f"Total rows: {n}")

data_hist = data.head(split_1)
data_train = data.slice(split_1, split_2 - split_1)
data_val = data.tail(n - split_2)

print(f"History: {len(data_hist)}, Train: {len(data_train)}, Val: {len(data_val)}")

Исходный размер: 89334605 строк
Размер после ограничения истории (5 на юзера): 3739556 строк
Total rows: 3739556
History: 2243733, Train: 747911, Val: 747912


In [5]:
from cg.candgen import IALSCandidateGenerator, GlobalPopularityGenerator, ArtistPopularityGenerator, RetrievalStage
from features.feature_manager import ItemStaticFeatureSource, UserStaticFeatureSource, IALSDotProductSource, FeatureManager
from main import RecSysPipeline

сделаем данные

In [6]:
data = data_orig.clone()

min_t = data["timestamp"].min()
max_t = data["timestamp"].max()
duration = max_t - min_t

t1 = min_t + int(duration * 0.75)
t2 = min_t + int(duration * 0.9)

# Разрезаем
data_A = data.filter(pl.col("timestamp") < t1)
data_B = data.filter((pl.col("timestamp") >= t1) & (pl.col("timestamp") < t2))
data_C = data.filter(pl.col("timestamp") >= t2)

print('A:', data_A.height)
print('B:', data_B.height)
print('C:', data_C.height)

A: 59670783
B: 16854532
C: 12809290


In [6]:
data_good_val_train = data_A.clone()  # <= 0.75
data_good_val_hist = data.filter(pl.col("timestamp") < t2)  # <= 0.9

print('good val train: ', data_good_val_train.height)
print('good val hist:  ', data_good_val_hist.height)

good val train:  59670783
good val hist:   76525315


In [7]:
t1 = min_t + int(duration * 0.93)
t2 = min_t + int(duration * 0.98)

# Разрезаем
data_A = data.filter(pl.col("timestamp") < t1)
data_B = data.filter((pl.col("timestamp") >= t1) & (pl.col("timestamp") < t2))
data_C = data.filter(pl.col("timestamp") >= t2)

print('A:', data_A.height)
print('B:', data_B.height)
print('C:', data_C.height)

A: 80304087
B: 6505221
C: 2525297


In [23]:
data_quick_val_train = data_A.clone()   # <= 0.93
data_quick_val_hist = data.filter(pl.col("timestamp") < t2)  # <= 0.98

print('quick val train: ', data_quick_val_train.height)
print('quick val hist:  ', data_quick_val_hist.height)

quick val train:  80304087
quick val hist:   86809308


In [24]:
ials_gen = IALSCandidateGenerator(factors=128, iterations=30, file_prefix='quick_val_hist')

retrieval = RetrievalStage(generators=[ials_gen])

In [25]:
user_stats = UserStaticFeatureSource()

feature_manager = FeatureManager(sources=[user_stats])

In [26]:
pipeline = RecSysPipeline(retrieval_stage=retrieval, feature_manager=feature_manager)

In [27]:
pipeline.fit_all(train_history_data=data_quick_val_hist.lazy())

=== Fitting Retrieval Stage ===
[ials] Fitting model...


100%|██████████| 30/30 [13:25<00:00, 26.87s/it]


[ials] Fit complete. Data moved to CPU and saved to fittingdata/cg/quick_val_hist_ials
=== Fitting Feature Manager ===
      Fit now: user_static
[user_static] Fitting...
=== Fit Complete ===
